# Lending Club EDA -- F10 -- Dataset Comparison, Population Drift & Distribution Shift

**Status: built.** See the cell map below for what's actually in this notebook.

## What this notebook covers

Distribution comparison across natural splits: early vintage (2013-2014) vs. late vintage (2016-2017) feature-by-feature, and (data permitting) a high-level accepted-vs-rejected-applications comparison using the separate rejected-loans raw file, to see how different the applicant pool that got approved looks from the one that got rejected.

## Where this fits

One of 14 category notebooks under `notebooks/02_eda/`, each covering one EDA
dimension in depth (see `notebooks/03_data_cleaning/` for the separate notebook
where any actual cleaning/imputation/encoding happens -- these EDA notebooks
are read-only against `data/02_interim/lendingclub.duckdb` and never modify
or clean the data themselves). Every code cell in a built notebook has a
markdown cell before it (what/why/how/expected) and a markdown cell after it
(what the real output means and what's next).


## Cell map

| # | What it does |
|---|---|
| 1 | Connect; define a reusable Population Stability Index (PSI) helper |
| 2 | PSI for `int_rate`: 2013 vs. 2017 -- has pricing drifted across the modeling window? |
| 3 | PSI for `grade` mix: 2013 vs. 2017 -- has the risk-grade composition drifted? |
| 4 | Is the windowed (2013-2017) population representative of the excluded years, on the features that matter most? |
| 5 | Synthesis -- what drift means for a model trained on this window |

## Cell 1 -- PSI helper

**What / why:** Population Stability Index (PSI) is the standard credit-risk
metric for quantifying how much a feature's distribution has shifted between
two populations (commonly training vs. production, or one time period vs.
another). PSI < 0.1 is conventionally "no significant shift," 0.1-0.25 is
"moderate shift, worth monitoring," and >0.25 is "significant shift." Writing
one correct implementation and reusing it across cells avoids drift-detection
bugs from ad-hoc recomputation.

**How:** bin a reference and comparison distribution into the same bin edges
(deciles of the reference), compute
`sum((comparison_pct - reference_pct) * ln(comparison_pct / reference_pct))`.

**Expect:** no output beyond a confirmation print -- tooling setup only.

In [1]:
import os, duckdb, pandas as pd, numpy as np
ASSETS_TABLES = "../../data/04_assets/tables"
ASSETS_PLOTS = "../../data/04_assets/plots"
os.makedirs(ASSETS_TABLES, exist_ok=True)
os.makedirs(ASSETS_PLOTS, exist_ok=True)
con = duckdb.connect("../../data/02_interim/lendingclub.duckdb", read_only=True)

def psi(reference, comparison, bins=10):
    ref = pd.Series(reference).dropna()
    comp = pd.Series(comparison).dropna()
    edges = np.percentile(ref, np.linspace(0, 100, bins+1))
    edges[0], edges[-1] = -np.inf, np.inf
    ref_counts, _ = np.histogram(ref, bins=edges)
    comp_counts, _ = np.histogram(comp, bins=edges)
    ref_pct = np.clip(ref_counts / len(ref), 1e-6, None)
    comp_pct = np.clip(comp_counts / len(comp), 1e-6, None)
    return float(np.sum((comp_pct - ref_pct) * np.log(comp_pct / ref_pct)))

print("helper ready: psi(reference, comparison, bins=10) -> PSI value")
print("PSI < 0.1: no significant shift | 0.1-0.25: moderate | > 0.25: significant")


helper ready: psi(reference, comparison, bins=10) -> PSI value
PSI < 0.1: no significant shift | 0.1-0.25: moderate | > 0.25: significant


**What the output shows:** the helper is defined, with the
standard interpretation thresholds printed for reference in every later cell.

**Next:** applying it to `int_rate` first -- since Lending Club periodically
repriced its risk model, checking whether the interest-rate distribution
itself shifted meaningfully between the start and end of the modeling
window.

## Cell 2 -- PSI for int_rate: 2013 vs. 2017

**What / why:** `int_rate` is Lending Club's own risk-based pricing output,
and pricing models get recalibrated over time. If `int_rate`'s distribution
shifted substantially between 2013 and 2017, a model trained across the full
window is implicitly learning from two different pricing regimes blended
together -- worth knowing explicitly rather than assuming pricing was static.

**How:** pull `int_rate` for 2013-origination and 2017-origination loans
separately, compute PSI with 2013 as the reference.

**Expect:** some drift is likely, given 5 years of platform evolution and
economic conditions -- the question is whether it's the "moderate" or
"significant" that a model builder should treat as effectively a regime
change.

In [2]:
rates_2013 = con.sql("SELECT TRY_CAST(int_rate AS DOUBLE) r FROM windowed WHERE substr(issue_d,-4)='2013'").df()["r"]
rates_2017 = con.sql("SELECT TRY_CAST(int_rate AS DOUBLE) r FROM windowed WHERE substr(issue_d,-4)='2017'").df()["r"]

psi_rate = psi(rates_2013, rates_2017)
print(f"int_rate 2013 median: {rates_2013.median():.2f}%, 2017 median: {rates_2017.median():.2f}%")
print(f"PSI (2013 -> 2017): {psi_rate:.3f}")


int_rate 2013 median: 14.33%, 2017 median: 12.74%
PSI (2013 -> 2017): 0.140


**What the output shows:**
```
int_rate 2013 median: 14.33%, 2017 median: 12.74%
PSI (2013 -> 2017): 0.140
```
PSI of 0.140 is in the moderate-shift range -- worth monitoring, though not severe enough to treat 2013 and 2017 as fundamentally different populations.

**Next:** checking whether the underlying risk-grade *mix* (not just pricing
within grades) also shifted -- did Lending Club originate a different
proportion of A/B/C/.../G loans in 2017 than in 2013?

## Cell 3 -- PSI for grade mix: 2013 vs. 2017

**What / why:** grade is a categorical risk tier, not a continuous variable
-- its "drift" is about whether the *proportion* of loans in each grade
shifted, which the numeric PSI formula still applies to directly using grade
categories as bins instead of quantiles.

**How:** compute grade proportions for 2013 and 2017 separately, apply the
PSI formula manually across the 7 grade categories (A-G).

**Expect:** notebook 06 already found the grade mix shifted toward riskier
grades (D/E growing) between 2013-2016 before pulling back in 2017 -- this
should show up here as a nontrivial PSI value, quantifying what notebook 06
showed visually.

In [3]:
grade_2013 = con.sql("SELECT grade, count(*) n FROM windowed WHERE substr(issue_d,-4)='2013' GROUP BY 1").df().set_index("grade")["n"]
grade_2017 = con.sql("SELECT grade, count(*) n FROM windowed WHERE substr(issue_d,-4)='2017' GROUP BY 1").df().set_index("grade")["n"]

all_grades = sorted(set(grade_2013.index) | set(grade_2017.index))
p2013 = np.clip((grade_2013.reindex(all_grades).fillna(0) / grade_2013.sum()).values, 1e-6, None)
p2017 = np.clip((grade_2017.reindex(all_grades).fillna(0) / grade_2017.sum()).values, 1e-6, None)
psi_grade = float(np.sum((p2017 - p2013) * np.log(p2017 / p2013)))

grade_mix = pd.DataFrame({"grade": all_grades, "2013_pct": p2013, "2017_pct": p2017})
print(grade_mix.to_string(index=False))
print(f"\nPSI (grade mix, 2013 -> 2017): {psi_grade:.3f}")
grade_mix.to_csv(os.path.join(ASSETS_TABLES, "eda10_grade_mix.csv"), index=False)


grade  2013_pct  2017_pct
    A  0.131146  0.158811
    B  0.327253  0.277585
    C  0.282855  0.324348
    D  0.152562  0.144820
    E  0.067201  0.060323
    F  0.032581  0.021551
    G  0.006402  0.012562

PSI (grade mix, 2013 -> 2017): 0.029


**What the output shows:**
```
grade  2013_pct  2017_pct
    A  0.131146  0.158811
    B  0.327253  0.277585
    C  0.282855  0.324348
    D  0.152562  0.144820
    E  0.067201  0.060323
    F  0.032581  0.021551
    G  0.006402  0.012562

PSI (grade mix, 2013 -> 2017): 0.029
```
PSI of 0.029 is actually below the conventional 0.1 no-significant-shift threshold --
a smaller effect than the visual year-over-year swings in notebook 06
suggested. The individual grade percentages did move (B and F/G shrank, A/C
grew), but by the PSI convention used industry-wide for population stability,
this is a modest compositional shift, not a dramatic one. Worth remembering
as a calibration point: not every visually-apparent trend translates into a
large PSI, and PSI is the more decision-relevant number for deciding whether
a model needs retraining or re-validation.

**Next:** checking the other side of drift -- not year-over-year within the
window, but whether the windowed population itself (2013-2017) differs
meaningfully from the years excluded from it.

## Cell 4 -- is the windowed population representative of the excluded years?

**What / why:** the modeling window excludes 2007-2012 and 2018. That
exclusion was justified by volume and right-censoring concerns (notebooks 06
and 08), not by the excluded years being fundamentally different borrowers.
Checking PSI on `int_rate` and `annual_inc` between the windowed population
and the excluded population directly tests whether that assumption holds, or
whether the excluded years also differ compositionally, not just in
reliability.

**How:** compute PSI for `int_rate` and `annual_inc`, windowed vs. excluded
(matured loans outside 2013-2017).

**Expect:** some difference is expected (2007-2012 was Lending Club's early,
smaller-scale period with a different borrower mix), but the magnitude
matters for how much weight to put on generalizing conclusions beyond the
window.

In [4]:
windowed_rates = con.sql("SELECT TRY_CAST(int_rate AS DOUBLE) r FROM windowed").df()["r"]
excluded_rates = con.sql("""
    SELECT TRY_CAST(int_rate AS DOUBLE) r FROM matured
    WHERE CAST(substr(issue_d,-4) AS INT) NOT BETWEEN 2013 AND 2017
""").df()["r"]
windowed_income = con.sql("SELECT TRY_CAST(annual_inc AS DOUBLE) r FROM windowed WHERE annual_inc IS NOT NULL").df()["r"]
excluded_income = con.sql("""
    SELECT TRY_CAST(annual_inc AS DOUBLE) r FROM matured
    WHERE CAST(substr(issue_d,-4) AS INT) NOT BETWEEN 2013 AND 2017 AND annual_inc IS NOT NULL
""").df()["r"]

psi_rate_excl = psi(windowed_rates, excluded_rates)
psi_income_excl = psi(windowed_income, excluded_income)
print(f"PSI int_rate (windowed vs. excluded years): {psi_rate_excl:.3f}")
print(f"PSI annual_inc (windowed vs. excluded years): {psi_income_excl:.3f}")


PSI int_rate (windowed vs. excluded years): 0.014
PSI annual_inc (windowed vs. excluded years): 0.012


**What the output shows:**
```
PSI int_rate (windowed vs. excluded years): 0.014
PSI annual_inc (windowed vs. excluded years): 0.012
```
Both PSI values fall below the 0.1 no-significant-shift threshold -- despite spanning very different eras of Lending Club's history (its earliest, smallest-scale years plus the right-censored 2018 vintage), the excluded population isn't compositionally very different from the windowed one on these two features. That's a reassuring result: the 2013-2017 window was excluded from the rest of the population mainly for reliability and censoring reasons (per notebooks 06 and 08), not because it's a fundamentally different type of borrower.

**Next:** pulling together what population drift means for how this
dataset's findings should be used in Phase 1.

## Cell 5 -- synthesis

**What / why:** drift findings are only useful if they connect back to a
concrete recommendation for modeling. Summarizing what was found and what it
implies closes that loop.

**How:** a short printed recap referencing the actual PSI values computed
above.

**Expect:** a compact table of every drift check run in this notebook.

In [5]:
summary = pd.DataFrame([
    {"comparison": "int_rate, 2013 vs 2017", "PSI": round(psi_rate, 3)},
    {"comparison": "grade mix, 2013 vs 2017", "PSI": round(psi_grade, 3)},
    {"comparison": "int_rate, windowed vs excluded years", "PSI": round(psi_rate_excl, 3)},
    {"comparison": "annual_inc, windowed vs excluded years", "PSI": round(psi_income_excl, 3)},
])
print(summary.to_string(index=False))
summary.to_csv(os.path.join(ASSETS_TABLES, "eda10_summary.csv"), index=False)


                            comparison   PSI
                int_rate, 2013 vs 2017 0.140
               grade mix, 2013 vs 2017 0.029
  int_rate, windowed vs excluded years 0.014
annual_inc, windowed vs excluded years 0.012


**What the output shows:**
```
comparison   PSI
                int_rate, 2013 vs 2017 0.140
               grade mix, 2013 vs 2017 0.029
  int_rate, windowed vs excluded years 0.014
annual_inc, windowed vs excluded years 0.012
```
The clearest signal is within-window: `int_rate` drifted moderately (PSI
0.140) across 2013-2017 even though the windowed population as
a whole isn't very different from the years excluded from it (PSI well under
0.1 on both features checked). That's a useful, specific implication for
Phase 1: origination year deserves consideration as either a model feature
or a validation-split dimension *within* the window, while the window
boundary itself (2013 vs. everything outside it) doesn't need the same
scrutiny -- it was already a defensible split on reliability grounds alone.

**Next:** notebook 11 checks a related but distinct question -- not drift
over time, but whether the windowed population is a representative *sample*
in the statistical sense (selection bias, coverage across states/income
bands), independent of the time dimension.